Canizales Morales Jesús Demetrio

Ramírez González Javier

Valdez García Alondra Alejandra.

# **Proyecto final**
# DARTS: Differentiable Architecture Search

**Objetivo de este *notebook*.**

En este *notebook* se implementa la replicación del artículo: *DARTS: Differentiable Architecture Search*, escrito por Hanxiao Liu, Karen Simonyan y Yiming Yang.

<br>

**Contenido**
1. Imports y verificación de GPU
2. Primitivas y operaciones
3. MixedOp y Cell
4. NetworkSearch (Supernet)
5. Architect (optimización binivel)
6. Utilidades
7. Configuración
8. DataLoaders
9. Modelo y optimizadores
10. Funciones de *train* y validación
11. Ejecutar búsqueda
12. Construcción de la arquitectura discreta final
13. Entrenamiento final desde cero
14. Ejecutar evaluación final
15. Visualizar genotipo
16. Ejecución integral del experimento
17. Visualizar arquitectura como grafo
18. Análisis completo de la arquitectura.

---
**Especificaciones de hardware**
---
* **Lenguaje usado:**
* **Sistema operativo:**
* **Máquina usada**:

## **1. Imports y verificación de GPU**

In [1]:
#!pip install graphviz -q   

In [2]:
# Importación de bibliotecas

import os
import time
import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, SubsetRandomSampler
from collections import namedtuple
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    cudnn.benchmark = True
    cudnn.enabled = True
    torch.cuda.set_device(0)

PyTorch: 2.10.0+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


## **2. Primitivas y operaciones**

In [3]:
PRIMITIVES = [
    'none',
    'max_pool_3x3',
    'avg_pool_3x3',
    'skip_connect',
    'sep_conv_3x3',
    'sep_conv_5x5',
    'dil_conv_3x3',
    'dil_conv_5x5',
]

Genotype = namedtuple('Genotype', 'normal normal_concat reduce reduce_concat')


class ReLUConvBN(nn.Module):
    def __init__(self, C_in, C_out, kernel_size, stride, padding, affine=True):
        super().__init__()
        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(C_in, C_out, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(C_out, affine=affine)
        )
    def forward(self, x):
        return self.op(x)


class DilConv(nn.Module):
    def __init__(self, C_in, C_out, kernel_size, stride, padding, dilation, affine=True):
        super().__init__()
        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(C_in, C_in, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=C_in, bias=False),
            nn.Conv2d(C_in, C_out, 1, padding=0, bias=False),
            nn.BatchNorm2d(C_out, affine=affine),
        )
    def forward(self, x):
        return self.op(x)


class SepConv(nn.Module):
    def __init__(self, C_in, C_out, kernel_size, stride, padding, affine=True):
        super().__init__()
        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(C_in, C_in, kernel_size, stride=stride, padding=padding, groups=C_in, bias=False),
            nn.Conv2d(C_in, C_in, 1, padding=0, bias=False),
            nn.BatchNorm2d(C_in, affine=affine),
            nn.ReLU(inplace=False),
            nn.Conv2d(C_in, C_in, kernel_size, stride=1, padding=padding, groups=C_in, bias=False),
            nn.Conv2d(C_in, C_out, 1, padding=0, bias=False),
            nn.BatchNorm2d(C_out, affine=affine),
        )
    def forward(self, x):
        return self.op(x)


class Identity(nn.Module):
    def forward(self, x):
        return x


class Zero(nn.Module):
    def __init__(self, stride):
        super().__init__()
        self.stride = stride
    def forward(self, x):
        if self.stride == 1:
            return x.mul(0.)
        return x[:, :, ::self.stride, ::self.stride].mul(0.)


class FactorizedReduce(nn.Module):
    def __init__(self, C_in, C_out, affine=True):
        super().__init__()
        assert C_out % 2 == 0
        self.relu = nn.ReLU(inplace=False)
        self.conv_1 = nn.Conv2d(C_in, C_out // 2, 1, stride=2, padding=0, bias=False)
        self.conv_2 = nn.Conv2d(C_in, C_out // 2, 1, stride=2, padding=0, bias=False)
        self.bn = nn.BatchNorm2d(C_out, affine=affine)
    def forward(self, x):
        x = self.relu(x)
        out = torch.cat([self.conv_1(x), self.conv_2(x[:, :, 1:, 1:])], dim=1)
        return self.bn(out)


OPS = {
    'none': lambda C, stride, affine: Zero(stride),
    'avg_pool_3x3': lambda C, stride, affine: nn.AvgPool2d(3, stride=stride, padding=1, count_include_pad=False),
    'max_pool_3x3': lambda C, stride, affine: nn.MaxPool2d(3, stride=stride, padding=1),
    'skip_connect': lambda C, stride, affine: Identity() if stride == 1 else FactorizedReduce(C, C, affine=affine),
    'sep_conv_3x3': lambda C, stride, affine: SepConv(C, C, 3, stride, 1, affine=affine),
    'sep_conv_5x5': lambda C, stride, affine: SepConv(C, C, 5, stride, 2, affine=affine),
    'dil_conv_3x3': lambda C, stride, affine: DilConv(C, C, 3, stride, 2, 2, affine=affine),
    'dil_conv_5x5': lambda C, stride, affine: DilConv(C, C, 5, stride, 4, 2, affine=affine),
}

print(f" {len(PRIMITIVES)} primitivas: {PRIMITIVES}")

 8 primitivas: ['none', 'max_pool_3x3', 'avg_pool_3x3', 'skip_connect', 'sep_conv_3x3', 'sep_conv_5x5', 'dil_conv_3x3', 'dil_conv_5x5']


##  **3. MixedOp y Cell**

In [4]:
class MixedOp(nn.Module):
    def __init__(self, C, stride):
        super().__init__()
        self._ops = nn.ModuleList()
        for prim in PRIMITIVES:
            op = OPS[prim](C, stride, False)
            if 'pool' in prim:
                op = nn.Sequential(op, nn.BatchNorm2d(C, affine=False))
            self._ops.append(op)

    def forward(self, x, weights):
        return sum(w * op(x) for w, op in zip(weights, self._ops))


class Cell(nn.Module):
    def __init__(self, steps, multiplier, C_prev_prev, C_prev, C, reduction, reduction_prev):
        super().__init__()
        self.reduction = reduction
        self._steps = steps
        self._multiplier = multiplier

        if reduction_prev:
            self.preprocess0 = FactorizedReduce(C_prev_prev, C, affine=False)
        else:
            self.preprocess0 = ReLUConvBN(C_prev_prev, C, 1, 1, 0, affine=False)
        self.preprocess1 = ReLUConvBN(C_prev, C, 1, 1, 0, affine=False)

        self._ops = nn.ModuleList()
        for i in range(self._steps):
            for j in range(2 + i):
                stride = 2 if reduction and j < 2 else 1
                self._ops.append(MixedOp(C, stride))

    def forward(self, s0, s1, weights):
        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)
        states = [s0, s1]
        offset = 0
        for i in range(self._steps):
            s = sum(self._ops[offset + j](h, weights[offset + j]) for j, h in enumerate(states))
            offset += len(states)
            states.append(s)
        return torch.cat(states[-self._multiplier:], dim=1)


##  **4. NetworkSearch (Supernet)**

In [5]:
class NetworkSearch(nn.Module):
    def __init__(self, C, num_classes, layers, criterion, steps=4, multiplier=4, stem_multiplier=3):
        super().__init__()
        self._C = C
        self._num_classes = num_classes
        self._layers = layers
        self._criterion = criterion
        self._steps = steps
        self._multiplier = multiplier

        C_curr = stem_multiplier * C
        self.stem = nn.Sequential(
            nn.Conv2d(3, C_curr, 3, padding=1, bias=False),
            nn.BatchNorm2d(C_curr)
        )

        C_prev_prev, C_prev, C_curr = C_curr, C_curr, C
        self.cells = nn.ModuleList()
        reduction_prev = False
        for i in range(layers):
            if i in [layers // 3, 2 * layers // 3]:
                C_curr *= 2
                reduction = True
            else:
                reduction = False
            cell = Cell(steps, multiplier, C_prev_prev, C_prev, C_curr, reduction, reduction_prev)
            reduction_prev = reduction
            self.cells.append(cell)
            C_prev_prev, C_prev = C_prev, multiplier * C_curr

        self.global_pooling = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(C_prev, num_classes)
        self._initialize_alphas()

    def _initialize_alphas(self):
        k = sum(1 for i in range(self._steps) for n in range(2 + i))
        num_ops = len(PRIMITIVES)
        self.alphas_normal = nn.Parameter(torch.randn(k, num_ops, device=DEVICE) * 1e-3)
        self.alphas_reduce = nn.Parameter(torch.randn(k, num_ops, device=DEVICE) * 1e-3)

    def arch_parameters(self):
        return [self.alphas_normal, self.alphas_reduce]

    def new(self):
        model_new = NetworkSearch(self._C, self._num_classes, self._layers, self._criterion).to(DEVICE)
        for x, y in zip(model_new.arch_parameters(), self.arch_parameters()):
            x.data.copy_(y.data)
        return model_new

    def forward(self, x):
        s0 = s1 = self.stem(x)
        for cell in self.cells:
            if cell.reduction:
                weights = F.softmax(self.alphas_reduce, dim=-1)
            else:
                weights = F.softmax(self.alphas_normal, dim=-1)
            s0, s1 = s1, cell(s0, s1, weights)
        out = self.global_pooling(s1)
        return self.classifier(out.view(out.size(0), -1))

    def _loss(self, x, target):
        return self._criterion(self(x), target)

    def genotype(self):
        def _parse(weights_np):
            gene = []
            n = 2
            start = 0
            none_idx = PRIMITIVES.index('none')
            for i in range(self._steps):
                end = start + n
                W = weights_np[start:end].copy()
                edges = sorted(
                    range(i + 2),
                    key=lambda x: -max(W[x][k] for k in range(len(W[x])) if k != none_idx)
                )[:2]
                for j in edges:
                    k_best = max(
                        range(len(W[j])),
                        key=lambda k: W[j][k] if k != none_idx else -1
                    )
                    gene.append((PRIMITIVES[k_best], j))
                start = end
                n += 1
            return gene

        gene_normal = _parse(F.softmax(self.alphas_normal, dim=-1).detach().cpu().numpy())
        gene_reduce = _parse(F.softmax(self.alphas_reduce, dim=-1).detach().cpu().numpy())
        concat = list(range(2 + self._steps - self._multiplier, self._steps + 2))
        return Genotype(
            normal=gene_normal, normal_concat=concat,
            reduce=gene_reduce, reduce_concat=concat
        )


##  **5. Architect (optimización binivel)**

In [6]:
class Architect:
    def __init__(self, model, args):
        self.network_momentum = args['momentum']
        self.network_weight_decay = args['weight_decay']
        self.model = model
        self.optimizer = torch.optim.Adam(
            model.arch_parameters(),
            lr=args['arch_learning_rate'],
            betas=(0.5, 0.999),
            weight_decay=args['arch_weight_decay']
        )

    def step(self, inp_trg, tgt_trg, inp_val, tgt_val, eta, net_opt, unrolled):
        self.optimizer.zero_grad()
        if unrolled:
            self._backward_step_unrolled(inp_trg, tgt_trg, inp_val, tgt_val, eta, net_opt)
        else:
            self._backward_step(inp_val, tgt_val)
        self.optimizer.step()

    def _backward_step(self, inp_val, tgt_val):
        loss = self.model._loss(inp_val, tgt_val)
        loss.backward()

    def _backward_step_unrolled(self, inp_trg, tgt_trg, inp_val, tgt_val, eta, net_opt):
        unrolled = self._compute_unrolled_model(inp_trg, tgt_trg, eta, net_opt)
        unrolled_loss = unrolled._loss(inp_val, tgt_val)
        unrolled_loss.backward()
        dalpha = [v.grad for v in unrolled.arch_parameters()]
        vector = [v.grad.data for v in unrolled.parameters()]
        implicit = self._hessian_vector_product(vector, inp_trg, tgt_trg)
        for g, ig in zip(dalpha, implicit):
            g.data.sub_(eta, ig.data)
        for v, g in zip(self.model.arch_parameters(), dalpha):
            if v.grad is None:
                v.grad = g.data.clone()
            else:
                v.grad.data.copy_(g.data)

    def _compute_unrolled_model(self, inp, tgt, eta, net_opt):
        loss = self.model._loss(inp, tgt)
        theta = torch.cat([p.data.flatten() for p in self.model.parameters()])
        try:
            moment = torch.cat([net_opt.state[v]['momentum_buffer'].flatten() for v in self.model.parameters()]).mul_(self.network_momentum)
        except:
            moment = torch.zeros_like(theta)
        grads = torch.autograd.grad(loss, self.model.parameters(), retain_graph=True)
        dtheta = torch.cat([g.flatten() for g in grads]).data + self.network_weight_decay * theta
        return self._construct_model_from_theta(theta.sub(eta, moment + dtheta))

    def _construct_model_from_theta(self, theta):
        model_new = self.model.new()
        model_dict = self.model.state_dict()
        offset = 0
        for k, v in self.model.named_parameters():
            vl = v.numel()
            model_dict[k] = theta[offset:offset + vl].view(v.size())
            offset += vl
        model_new.load_state_dict(model_dict)
        return model_new

    def _hessian_vector_product(self, vector, inp, tgt, r=1e-2):
        R = r / torch.cat([v.flatten() for v in vector]).norm()
        for p, v in zip(self.model.parameters(), vector):
            p.data.add_(R, v)
        grads_p = torch.autograd.grad(self.model._loss(inp, tgt), self.model.arch_parameters())
        for p, v in zip(self.model.parameters(), vector):
            p.data.sub_(2 * R, v)
        grads_n = torch.autograd.grad(self.model._loss(inp, tgt), self.model.arch_parameters())
        for p, v in zip(self.model.parameters(), vector):
            p.data.add_(R, v)
        return [(x - y).div_(2 * R) for x, y in zip(grads_p, grads_n)]


##  **6. Utilidades**

In [7]:
class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.avg = 0
        self.sum = 0
        self.cnt = 0
    def update(self, val, n=1):
        self.sum += val * n
        self.cnt += n
        self.avg = self.sum / self.cnt


def accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return tuple(res)


class Cutout:
    def __init__(self, length):
        self.length = length
    def __call__(self, img):
        h, w = img.size(1), img.size(2)
        mask = torch.ones(h, w, dtype=torch.float32)
        y = torch.randint(0, h, (1,)).item()
        x = torch.randint(0, w, (1,)).item()
        y1 = max(0, y - self.length // 2)
        y2 = min(h, y + self.length // 2)
        x1 = max(0, x - self.length // 2)
        x2 = min(w, x + self.length // 2)
        mask[y1:y2, x1:x2] = 0.
        img *= mask.unsqueeze(0).expand_as(img)
        return img


def count_params_in_MB(model):
    return sum(v.numel() for name, v in model.named_parameters() if 'auxiliary' not in name) / 1e6


##  **7. Configuración**

In [8]:
SEARCH_CFG = {
    "dataset": "CIFAR10",
    "batch_size": 64,
    "learning_rate": 0.025,
    "learning_rate_min": 0.001,
    "momentum": 0.9,
    "weight_decay": 3e-4,
    "arch_learning_rate": 3e-4,
    "arch_weight_decay": 1e-3,
    "epochs": 50,
    "init_channels": 16,
    "layers": 8,
    "steps": 4,
    "multiplier": 4,
    "stem_multiplier": 3,
    "train_portion": 0.5,
    "cutout": False,
    "cutout_length": 16,
    "grad_clip": 5,
    "unrolled": False,
    "num_workers": 1,
    "seed": 11,
    "data": "../data",
    "report_freq": 50,
    "arch_update_freq": 1
}

In [9]:
EVAL_CFG = {
    "dataset": "CIFAR10",
    "batch_size": 64,
    "learning_rate": 0.025,
    "momentum": 0.9,
    "weight_decay": 3e-4,
    "epochs": 600,
    "init_channels": 36,
    "layers": 20,
    "auxiliary": False,
    "cutout": False, # Antes True
    "cutout_length": 16,
    "drop_path_prob": 0.0, # Antes 0.2
    "num_workers": 1,
    "seed": 11,
    "data": "../data",
    "stem_multiplier": 3,
    "report_freq": 50,
    "grad_clip": 5,
    "learning_rate_min": 0.0
}

In [10]:
# Flags de ejecución (pruebas rápidas)

RUN_QUICK_CHECKS = False
RUN_SEARCH_EXAMPLE = False
RUN_EVAL_EXAMPLE = False
RUN_FULL_EXPERIMENT = False
RUN_GENOTYPE_VIS = False
RUN_FINAL_ANALYSIS = False
RUN_MULTI_SEED = False
RUN_SYSTEM_INFO = False

EXPERIMENT_SEEDS = [11, 21, 32, 42]

# Validación de modos de ejecución
if RUN_FULL_EXPERIMENT and RUN_MULTI_SEED:
    raise ValueError("No actives RUN_FULL_EXPERIMENT y RUN_MULTI_SEED al mismo tiempo.")

if RUN_EVAL_EXAMPLE and not RUN_SEARCH_EXAMPLE:
    raise ValueError("RUN_EVAL_EXAMPLE requiere RUN_SEARCH_EXAMPLE=False.")

In [11]:
# Configuraciones de comparación DARTS

SEARCH_CFG_FIRST_ORDER = copy.deepcopy(SEARCH_CFG)
SEARCH_CFG_FIRST_ORDER["unrolled"] = False

SEARCH_CFG_SECOND_ORDER = copy.deepcopy(SEARCH_CFG)
SEARCH_CFG_SECOND_ORDER["unrolled"] = True

In [12]:
def set_reproducibility(seed: int, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    print(f"[Reproducibility] Seed={seed} | deterministic={deterministic}")

##  **8. DataLoaders**

In [13]:
def get_cifar10_transforms(use_cutout=False, cutout_length=16):
    CIFAR_MEAN = [0.49139968, 0.48215827, 0.44653124]
    CIFAR_STD = [0.24703233, 0.24348505, 0.26158768]

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    if use_cutout:
        train_tf.transforms.append(Cutout(cutout_length))

    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    return train_tf, eval_tf


def build_search_dataloaders(search_cfg):
    """
    Construye los DataLoaders para la fase de búsqueda (search).
    Usa únicamente el train oficial de CIFAR-10 y lo divide en
    train_search y valid_search.
    """
    train_tf, _ = get_cifar10_transforms(
        use_cutout=search_cfg["cutout"],
        cutout_length=search_cfg["cutout_length"]
    )

    train_data = dset.CIFAR10(
        root=search_cfg["data"],
        train=True,
        download=True,
        transform=train_tf
    )

    num_train = len(train_data)
    indices = list(range(num_train))
    split = int(np.floor(search_cfg["train_portion"] * num_train))

    rng = np.random.RandomState(search_cfg["seed"])
    rng.shuffle(indices)

    train_indices = indices[:split]
    valid_indices = indices[split:]

    train_queue = DataLoader(
        train_data,
        batch_size=search_cfg["batch_size"],
        sampler=SubsetRandomSampler(train_indices),
        num_workers=search_cfg["num_workers"],
        pin_memory=True
    )

    valid_queue = DataLoader(
        train_data,
        batch_size=search_cfg["batch_size"],
        sampler=SubsetRandomSampler(valid_indices),
        num_workers=search_cfg["num_workers"],
        pin_memory=True
    )

    print("=" * 70)
    print("SEARCH DATALOADERS")
    print("=" * 70)
    print(f"Train-search samples: {len(train_indices)} | batches: {len(train_queue)}")
    print(f"Valid-search samples: {len(valid_indices)} | batches: {len(valid_queue)}")

    return train_queue, valid_queue


def build_eval_dataloaders(eval_cfg, use_validation_split=True):
    """
    Construye DataLoaders para evaluación final.

    Si use_validation_split=True:
        - divide el train oficial en train_final y val_final
        - train usa train_tf
        - val usa eval_tf
        - test usa eval_tf

    Si False:
        - usa todo train para entrenar
        - devuelve val_queue = None
    """
    train_tf, eval_tf = get_cifar10_transforms(
        use_cutout=eval_cfg["cutout"],
        cutout_length=eval_cfg["cutout_length"]
    )

    # Dataset para entrenamiento final (con augmentations)
    train_data = dset.CIFAR10(
        root=eval_cfg["data"],
        train=True,
        download=True,
        transform=train_tf
    )

    # Dataset para validación final (sin augmentations)
    val_data = dset.CIFAR10(
        root=eval_cfg["data"],
        train=True,
        download=True,
        transform=eval_tf
    )

    # Dataset de test oficial (sin augmentations)
    test_data = dset.CIFAR10(
        root=eval_cfg["data"],
        train=False,
        download=True,
        transform=eval_tf
    )

    num_train = len(train_data)
    indices = list(range(num_train))

    if use_validation_split:
        split = int(0.8 * num_train)

        rng = np.random.RandomState(eval_cfg["seed"])
        rng.shuffle(indices)

        train_indices = indices[:split]
        val_indices = indices[split:]

        train_queue = DataLoader(
            train_data,
            batch_size=eval_cfg["batch_size"],
            sampler=SubsetRandomSampler(train_indices),
            num_workers=eval_cfg["num_workers"],
            pin_memory=True
        )

        val_queue = DataLoader(
            val_data,
            batch_size=eval_cfg["batch_size"],
            sampler=SubsetRandomSampler(val_indices),
            num_workers=eval_cfg["num_workers"],
            pin_memory=True
        )

        print("=" * 70)
        print("EVAL DATALOADERS (train/val/test)")
        print("=" * 70)
        print(f"Train: {len(train_indices)}")
        print(f"Val:   {len(val_indices)}")
        print(f"Test:  {len(test_data)}")

    else:
        train_queue = DataLoader(
            train_data,
            batch_size=eval_cfg["batch_size"],
            shuffle=True,
            num_workers=eval_cfg["num_workers"],
            pin_memory=True
        )
        val_queue = None

        print("=" * 70)
        print("EVAL DATALOADERS (train/test)")
        print("=" * 70)
        print(f"Train: {len(train_data)}")
        print(f"Test:  {len(test_data)}")

    test_queue = DataLoader(
        test_data,
        batch_size=eval_cfg["batch_size"],
        shuffle=False,
        num_workers=eval_cfg["num_workers"],
        pin_memory=True
    )

    return train_queue, val_queue, test_queue

In [14]:
if RUN_QUICK_CHECKS:
    # Pruebas rápidas de construcción de DataLoaders
    search_train_queue, search_valid_queue = build_search_dataloaders(SEARCH_CFG)

    eval_train_queue, eval_val_queue, eval_test_queue = build_eval_dataloaders(
        EVAL_CFG,
        use_validation_split=True
    )

##  **9. Modelo y optimizadores**

In [15]:
def build_criterion():
  '''
  Función para construir el criterio.
  '''
  return nn.CrossEntropyLoss().to(DEVICE)


def build_search_components(search_cfg, num_classes=10):
    """
    Construye todos los componentes necesarios para la fase de búsqueda (search):
    - criterion
    - supernet NetworkSearch
    - optimizer de pesos w
    - scheduler
    - architect (optimización de alpha)
    """
    criterion = build_criterion()

    model = NetworkSearch(
        C=search_cfg["init_channels"],
        num_classes=num_classes,
        layers=search_cfg["layers"],
        criterion=criterion,
        steps=search_cfg["steps"],
        multiplier=search_cfg["multiplier"],
        stem_multiplier=search_cfg["stem_multiplier"]
    ).to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=search_cfg["learning_rate"],
        momentum=search_cfg["momentum"],
        weight_decay=search_cfg["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=search_cfg["epochs"],
        eta_min=search_cfg["learning_rate_min"]
    )

    architect = Architect(model, search_cfg)

    print("=" * 70)
    print("SEARCH MODEL COMPONENTS")
    print("=" * 70)
    print(f"Model params: {count_params_in_MB(model):.3f} MB")
    print(f"Init channels: {search_cfg['init_channels']}")
    print(f"Layers: {search_cfg['layers']}")
    print(f"Steps: {search_cfg['steps']}")
    print(f"Multiplier: {search_cfg['multiplier']}")
    print(f"Stem multiplier: {search_cfg['stem_multiplier']}")
    print(f"Unrolled: {search_cfg['unrolled']}")

    return {
        "criterion": criterion,
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "architect": architect,
    }

Cabe la mención de que esta sección construye únicamente los componentes de `SEARCH`.

La red discreta y sus optimizadores para `EVALUATION FINAL` se agregarán después.

In [16]:
if RUN_QUICK_CHECKS:
    # Prueba rápida de construcción de componentes de search
    search_components = build_search_components(SEARCH_CFG, num_classes=10)

    search_criterion = search_components["criterion"]
    search_model = search_components["model"]
    search_optimizer = search_components["optimizer"]
    search_scheduler = search_components["scheduler"]
    search_architect = search_components["architect"]

##  **10. Funciones de entrenamiento y validación**

In [17]:
def train_search_epoch(
    train_q,
    valid_q,
    model,
    architect,
    criterion,
    optimizer,
    lr,
    search_cfg
):
    """
    Ejecuta una época de entrenamiento para la fase de búsqueda (search) de DARTS.

    En cada iteración:
    - usa un batch de train_q para actualizar los pesos w
    - usa un batch de valid_q para actualizar la arquitectura alpha
    """
    objs = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    model.train()
    valid_iter = iter(valid_q)

    for step, (inp, tgt) in enumerate(train_q):
        n = inp.size(0)

        inp = inp.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        # batch de validación para actualización de arquitectura
        try:
            inp_search, tgt_search = next(valid_iter)
        except StopIteration:
            valid_iter = iter(valid_q)
            inp_search, tgt_search = next(valid_iter)

        inp_search = inp_search.to(DEVICE, non_blocking=True)
        tgt_search = tgt_search.to(DEVICE, non_blocking=True)

        # Actualización de alpha
        if step % search_cfg.get("arch_update_freq", 1) == 0:
            architect.step(
                inp, tgt,
                inp_search, tgt_search,
                lr,
                optimizer,
                search_cfg["unrolled"]
            )


        # Actualización de pesos w
        optimizer.zero_grad(set_to_none=True)

        logits = model(inp)
        loss = criterion(logits, tgt)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), search_cfg["grad_clip"])
        optimizer.step()

        # Métricas
        prec1, prec5 = accuracy(logits, tgt, topk=(1, 5))

        objs.update(loss.item(), n)
        top1.update(prec1.item(), n)
        top5.update(prec5.item(), n)

        if step % search_cfg["report_freq"] == 0:
            print(
                f"  [train-search] Step {step:03d}/{len(train_q):03d} | "
                f"loss={objs.avg:.4f} | top1={top1.avg:.2f}% | top5={top5.avg:.2f}%"
            )

    return {
        "loss": objs.avg,
        "top1": top1.avg,
        "top5": top5.avg,
    }


@torch.no_grad()
def validate_search(valid_q, model, criterion, search_cfg=None):
    """
    Evalúa el supernet durante la fase de búsqueda.
    Esta validación sirve para monitorear el desempeño de la arquitectura
    continua durante search..
    """
    objs = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    model.eval()

    for step, (inp, tgt) in enumerate(valid_q):
        inp = inp.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        logits = model(inp)
        loss = criterion(logits, tgt)

        prec1, prec5 = accuracy(logits, tgt, topk=(1, 5))
        n = inp.size(0)

        objs.update(loss.item(), n)
        top1.update(prec1.item(), n)
        top5.update(prec5.item(), n)

        if search_cfg is not None and step % search_cfg["report_freq"] == 0:
            print(
                f"  [valid-search] Step {step:03d}/{len(valid_q):03d} | "
                f"loss={objs.avg:.4f} | top1={top1.avg:.2f}% | top5={top5.avg:.2f}%"
            )

    return {
        "loss": objs.avg,
        "top1": top1.avg,
        "top5": top5.avg,
    }

In [18]:
if RUN_QUICK_CHECKS:
    # Prueba rápida de funciones de search
    current_lr = search_scheduler.get_last_lr()[0]

    train_metrics = train_search_epoch(
        train_q=search_train_queue,
        valid_q=search_valid_queue,
        model=search_model,
        architect=search_architect,
        criterion=search_criterion,
        optimizer=search_optimizer,
        lr=current_lr,
        search_cfg=SEARCH_CFG
    )

    valid_metrics = validate_search(
        valid_q=search_valid_queue,
        model=search_model,
        criterion=search_criterion,
        search_cfg=SEARCH_CFG
    )

    print("\nResumen prueba rápida SEARCH:")
    print(train_metrics)
    print(valid_metrics)

## **11. Ejecutar búsqueda**

In [19]:
def genotype_to_dict(genotype):
    return {
        "normal": [(op, int(idx)) for op, idx in genotype.normal],
        "normal_concat": [int(x) for x in genotype.normal_concat],
        "reduce": [(op, int(idx)) for op, idx in genotype.reduce],
        "reduce_concat": [int(x) for x in genotype.reduce_concat],
    }

In [20]:
def run_search(search_cfg, algo_tag="first_order", num_classes=10):
    """
    Ejecuta la fase completa de búsqueda (search) de DARTS en CIFAR-10.
    """
    set_reproducibility(search_cfg["seed"], deterministic=False)

    run_dir = os.path.join(
        "results",
        "cifar10",
        "search",
        algo_tag,
        f"seed_{search_cfg['seed']}"
    )
    os.makedirs(run_dir, exist_ok=True)

    # Guardar configuración del experimento
    with open(os.path.join(run_dir, "config_search.json"), "w") as f:
        json.dump(search_cfg, f, indent=2)

    # Dataloaders
    train_queue, valid_queue = build_search_dataloaders(search_cfg)

    # Componentes de search
    search_components = build_search_components(search_cfg, num_classes=num_classes)
    criterion = search_components["criterion"]
    model = search_components["model"]
    optimizer = search_components["optimizer"]
    scheduler = search_components["scheduler"]
    architect = search_components["architect"]

    print("\n" + "=" * 70)
    print(f"DARTS SEARCH — CIFAR-10 | {algo_tag} | seed={search_cfg['seed']}")
    print("=" * 70)
    print(f"Epochs: {search_cfg['epochs']}")
    print(f"Init channels: {search_cfg['init_channels']}")
    print(f"Layers: {search_cfg['layers']}")
    print(f"Unrolled: {search_cfg['unrolled']}")
    print("=" * 70)

    history = []
    best_val_acc = -1.0
    best_genotype = None

    t0 = time.time()

    for epoch in range(search_cfg["epochs"]):
        epoch_start = time.time()
        lr = scheduler.get_last_lr()[0]

        print(f"\nEpoch {epoch+1}/{search_cfg['epochs']} | LR={lr:.6f}")
        print("-" * 70)

        genotype = model.genotype()
        print(f"Genotype (epoch start): {genotype}")

        train_metrics = train_search_epoch(
            train_q=train_queue,
            valid_q=valid_queue,
            model=model,
            architect=architect,
            criterion=criterion,
            optimizer=optimizer,
            lr=lr,
            search_cfg=search_cfg
        )

        valid_metrics = validate_search(
            valid_q=valid_queue,
            model=model,
            criterion=criterion,
            search_cfg=search_cfg
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        epoch_record = {
            "epoch": epoch + 1,
            "lr": float(lr),
            "train_loss": float(train_metrics["loss"]),
            "train_top1": float(train_metrics["top1"]),
            "train_top5": float(train_metrics["top5"]),
            "valid_loss": float(valid_metrics["loss"]),
            "valid_top1": float(valid_metrics["top1"]),
            "valid_top5": float(valid_metrics["top5"]),
            "epoch_time_sec": float(epoch_time),
            "genotype": str(genotype),
        }
        history.append(epoch_record)

        print(f"Train  | loss={train_metrics['loss']:.4f} | top1={train_metrics['top1']:.2f}% | top5={train_metrics['top5']:.2f}%")
        print(f"Valid  | loss={valid_metrics['loss']:.4f} | top1={valid_metrics['top1']:.2f}% | top5={valid_metrics['top5']:.2f}%")
        print(f"Time   | {epoch_time:.1f} sec")

        # Guardar mejor genotipo según valid_top1
        if valid_metrics["top1"] > best_val_acc:
            best_val_acc = valid_metrics["top1"]
            best_genotype = genotype

            torch.save(
                {
                    "epoch": epoch + 1,
                    "genotype": genotype_to_dict(genotype),
                    "alphas_normal": model.alphas_normal.detach().cpu(),
                    "alphas_reduce": model.alphas_reduce.detach().cpu(),
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_acc": best_val_acc,
                },
                os.path.join(run_dir, "best_search_checkpoint.pt")
            )

            with open(os.path.join(run_dir, "best_genotype.json"), "w") as f:
                json.dump(genotype_to_dict(genotype), f, indent=2)

            with open(os.path.join(run_dir, "best_genotype.txt"), "w") as f:
                f.write(str(genotype))

            print(f"★ Nuevo mejor genotipo guardado | valid_top1={best_val_acc:.2f}%")

        # Guardar historial tras cada época
        with open(os.path.join(run_dir, "search_history.json"), "w") as f:
            json.dump(history, f, indent=2)

    total_time = time.time() - t0

    summary = {
        "algo_tag": algo_tag,
        "seed": int(search_cfg["seed"]),
        "best_valid_top1": float(best_val_acc),
        "total_search_time_sec": float(total_time),
        "run_dir": run_dir,
        "best_genotype": genotype_to_dict(best_genotype) if best_genotype is not None else None,
    }

    with open(os.path.join(run_dir, "search_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n" + "=" * 70)
    print("SEARCH FINALIZADO")
    print("=" * 70)
    print(f"Best valid top1: {best_val_acc:.2f}%")
    print(f"Tiempo total: {total_time/60:.2f} min")
    print(f"Resultados en: {run_dir}")
    print("=" * 70)

    return summary

In [21]:
if RUN_SEARCH_EXAMPLE:
    # Ejemplo de corrida de búsqueda
    search_summary = run_search(
        search_cfg=SEARCH_CFG,
        algo_tag="first_order",
        num_classes=10
    )

    print("\nResumen de búsqueda:")
    print(search_summary)

## **12. Construcción de la arquitectura discreta final**

In [22]:
def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    if drop_prob == 0.0 or not training:
        return x

    keep_prob = 1.0 - drop_prob
    shape = (x.size(0), 1, 1, 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    x = x.div(keep_prob) * random_tensor
    return x


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

In [23]:
OPS_FINAL = {
    'none': lambda C, stride, affine: Zero(stride),
    'avg_pool_3x3': lambda C, stride, affine: nn.AvgPool2d(
        3, stride=stride, padding=1, count_include_pad=False
    ),
    'max_pool_3x3': lambda C, stride, affine: nn.MaxPool2d(
        3, stride=stride, padding=1
    ),
    'skip_connect': lambda C, stride, affine: Identity() if stride == 1 else FactorizedReduce(C, C, affine=affine),
    'sep_conv_3x3': lambda C, stride, affine: SepConv(C, C, 3, stride, 1, affine=affine),
    'sep_conv_5x5': lambda C, stride, affine: SepConv(C, C, 5, stride, 2, affine=affine),
    'dil_conv_3x3': lambda C, stride, affine: DilConv(C, C, 3, stride, 2, 2, affine=affine),
    'dil_conv_5x5': lambda C, stride, affine: DilConv(C, C, 5, stride, 4, 2, affine=affine),
}

In [24]:
class CellOp(nn.Module):
    def __init__(self, op_name, C, stride):
        super().__init__()
        self.op_name = op_name
        self.op = OPS_FINAL[op_name](C, stride, True)

        if 'pool' in op_name:
            self.op = nn.Sequential(self.op, nn.BatchNorm2d(C, affine=True))

    def forward(self, x):
        return self.op(x)


class CellDiscrete(nn.Module):
    def __init__(
        self,
        genotype,
        C_prev_prev,
        C_prev,
        C,
        reduction,
        reduction_prev,
        drop_path_prob=0.0
    ):
        super().__init__()

        self.reduction = reduction
        self.drop_path_prob = drop_path_prob

        if reduction_prev:
            self.preprocess0 = FactorizedReduce(C_prev_prev, C, affine=True)
        else:
            self.preprocess0 = ReLUConvBN(C_prev_prev, C, 1, 1, 0, affine=True)

        self.preprocess1 = ReLUConvBN(C_prev, C, 1, 1, 0, affine=True)

        if reduction:
            op_names, indices = zip(*genotype.reduce)
            concat = genotype.reduce_concat
        else:
            op_names, indices = zip(*genotype.normal)
            concat = genotype.normal_concat

        self._compile(C, op_names, indices, concat, reduction)

    def _compile(self, C, op_names, indices, concat, reduction):
        self._steps = len(op_names) // 2
        self._concat = concat
        self.multiplier = len(concat)
        self._indices = indices
        self._op_names = op_names

        self._ops = nn.ModuleList()
        for name, index in zip(op_names, indices):
            stride = 2 if reduction and index < 2 else 1
            op = CellOp(name, C, stride)
            self._ops.append(op)

    def forward(self, s0, s1):
        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)

        states = [s0, s1]
        offset = 0

        for _ in range(self._steps):
            h1 = states[self._indices[offset]]
            h2 = states[self._indices[offset + 1]]

            op1 = self._ops[offset]
            op2 = self._ops[offset + 1]

            h1 = op1(h1)
            h2 = op2(h2)

            if self.training and self.drop_path_prob > 0.0:
                if op1.op_name != 'skip_connect':
                    h1 = drop_path(h1, self.drop_path_prob, self.training)
                if op2.op_name != 'skip_connect':
                    h2 = drop_path(h2, self.drop_path_prob, self.training)

            s = h1 + h2
            states.append(s)
            offset += 2

        return torch.cat([states[i] for i in self._concat], dim=1)

In [25]:
class AuxiliaryHeadCIFAR(nn.Module):
    def __init__(self, C, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.AvgPool2d(5, stride=3, padding=0, count_include_pad=False),
            nn.Conv2d(C, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 768, 2, bias=False),
            nn.BatchNorm2d(768),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [26]:
'''
Esta es la red final que se entrenará desde cero.
'''

class NetworkCIFARFinal(nn.Module):
    def __init__(
        self,
        C,
        num_classes,
        layers,
        auxiliary,
        genotype,
        drop_path_prob=0.0,
        stem_multiplier=3
    ):
        super().__init__()

        self._layers = layers
        self._auxiliary = auxiliary
        self.drop_path_prob = drop_path_prob

        C_curr = stem_multiplier * C
        self.stem = nn.Sequential(
            nn.Conv2d(3, C_curr, 3, padding=1, bias=False),
            nn.BatchNorm2d(C_curr)
        )

        C_prev_prev, C_prev, C_curr = C_curr, C_curr, C
        self.cells = nn.ModuleList()
        reduction_prev = False
        self.auxiliary_head_index = 2 * layers // 3 if auxiliary else -1

        for i in range(layers):
            if i in [layers // 3, 2 * layers // 3]:
                C_curr *= 2
                reduction = True
            else:
                reduction = False

            cell = CellDiscrete(
                genotype=genotype,
                C_prev_prev=C_prev_prev,
                C_prev=C_prev,
                C=C_curr,
                reduction=reduction,
                reduction_prev=reduction_prev,
                drop_path_prob=self.drop_path_prob
            )

            reduction_prev = reduction
            self.cells.append(cell)
            C_prev_prev, C_prev = C_prev, cell.multiplier * C_curr

            if i == self.auxiliary_head_index:
                C_to_auxiliary = C_prev

        if auxiliary:
            self.auxiliary_head = AuxiliaryHeadCIFAR(C_to_auxiliary, num_classes)
        else:
            self.auxiliary_head = None

        self.global_pooling = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(C_prev, num_classes)

    def forward(self, x):
        logits_aux = None
        s0 = s1 = self.stem(x)

        for i, cell in enumerate(self.cells):
            s0, s1 = s1, cell(s0, s1)

            if self._auxiliary and self.training and i == self.auxiliary_head_index:
                logits_aux = self.auxiliary_head(s1)

        out = self.global_pooling(s1)
        logits = self.classifier(out.view(out.size(0), -1))

        if self._auxiliary and self.training:
            return logits, logits_aux

        return logits

In [27]:
def build_final_model(eval_cfg, genotype, num_classes=10):
    '''
    Función para construir la red final.
    '''
    model = NetworkCIFARFinal(
        C=eval_cfg["init_channels"],
        num_classes=num_classes,
        layers=eval_cfg["layers"],
        auxiliary=eval_cfg["auxiliary"],
        genotype=genotype,
        drop_path_prob=eval_cfg["drop_path_prob"],
        stem_multiplier=eval_cfg.get("stem_multiplier", 3)
    ).to(DEVICE)

    print("=" * 70)
    print("FINAL DISCRETE MODEL")
    print("=" * 70)
    print(f"Model params: {count_params_in_MB(model):.3f} MB")
    print(f"Init channels: {eval_cfg['init_channels']}")
    print(f"Layers: {eval_cfg['layers']}")
    print(f"Auxiliary: {eval_cfg['auxiliary']}")
    print(f"Drop path prob: {eval_cfg['drop_path_prob']}")

    return model

In [28]:
# Para reconstruir el genotipo desde el resultado de search

def dict_to_genotype(genotype_dict):
    return Genotype(
        normal=[tuple(x) for x in genotype_dict["normal"]],
        normal_concat=list(genotype_dict["normal_concat"]),
        reduce=[tuple(x) for x in genotype_dict["reduce"]],
        reduce_concat=list(genotype_dict["reduce_concat"]),
    )

In [29]:
if RUN_QUICK_CHECKS and RUN_SEARCH_EXAMPLE:
    # Prueba rápida: construir genotipo y modelo final a partir del search ya corrido
    best_genotype = dict_to_genotype(search_summary["best_genotype"])

    final_model = build_final_model(
        eval_cfg=EVAL_CFG,
        genotype=best_genotype,
        num_classes=10
    )

## **13. Entrenamiento final desde cero**

In [30]:
def build_final_optimizer_scheduler(model, eval_cfg):
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=eval_cfg["learning_rate"],
        momentum=eval_cfg["momentum"],
        weight_decay=eval_cfg["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=eval_cfg["epochs"],
        eta_min=eval_cfg.get("learning_rate_min", 0.0)
    )

    return optimizer, scheduler


def build_final_components(eval_cfg, genotype, num_classes=10):
    criterion = build_criterion()

    model = build_final_model(
        eval_cfg=eval_cfg,
        genotype=genotype,
        num_classes=num_classes
    )

    optimizer, scheduler = build_final_optimizer_scheduler(model, eval_cfg)

    print("=" * 70)
    print("FINAL TRAINING COMPONENTS")
    print("=" * 70)
    print(f"Learning rate: {eval_cfg['learning_rate']}")
    print(f"Momentum: {eval_cfg['momentum']}")
    print(f"Weight decay: {eval_cfg['weight_decay']}")
    print(f"Epochs: {eval_cfg['epochs']}")

    return {
        "criterion": criterion,
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
    }

In [31]:
def train_final_epoch(train_q, model, criterion, optimizer, eval_cfg, epoch=None):
    objs = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    model.train()

    if hasattr(model, "drop_path_prob"):
      if epoch is not None and eval_cfg["epochs"] > 1:
          current_drop_path = eval_cfg["drop_path_prob"] * epoch / (eval_cfg["epochs"] - 1)
      else:
          current_drop_path = eval_cfg["drop_path_prob"]

      model.drop_path_prob = current_drop_path

      # Propagar a cada celda discreta
      if hasattr(model, "cells"):
          for cell in model.cells:
              if hasattr(cell, "drop_path_prob"):
                  cell.drop_path_prob = current_drop_path

    for step, (inp, tgt) in enumerate(train_q):
        n = inp.size(0)

        inp = inp.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(inp)

        if isinstance(logits, tuple):
            logits, logits_aux = logits
            loss = criterion(logits, tgt) + 0.4 * criterion(logits_aux, tgt)
        else:
            loss = criterion(logits, tgt)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), eval_cfg["grad_clip"])
        optimizer.step()

        prec1, prec5 = accuracy(logits, tgt, topk=(1, 5))

        objs.update(loss.item(), n)
        top1.update(prec1.item(), n)
        top5.update(prec5.item(), n)

        if step % eval_cfg["report_freq"] == 0:
            print(
                f"  [train-final] Step {step:03d}/{len(train_q):03d} | "
                f"loss={objs.avg:.4f} | top1={top1.avg:.2f}% | top5={top5.avg:.2f}%"
            )

    return {
        "loss": objs.avg,
        "top1": top1.avg,
        "top5": top5.avg,
    }

In [32]:
@torch.no_grad()
def evaluate_final(data_q, model, criterion, eval_cfg=None, split_name="test"):
    objs = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    model.eval()

    for step, (inp, tgt) in enumerate(data_q):
        inp = inp.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        logits = model(inp)
        if isinstance(logits, tuple):
            logits = logits[0]

        loss = criterion(logits, tgt)
        prec1, prec5 = accuracy(logits, tgt, topk=(1, 5))
        n = inp.size(0)

        objs.update(loss.item(), n)
        top1.update(prec1.item(), n)
        top5.update(prec5.item(), n)

        if eval_cfg is not None and step % eval_cfg["report_freq"] == 0:
            print(
                f"  [{split_name}] Step {step:03d}/{len(data_q):03d} | "
                f"loss={objs.avg:.4f} | top1={top1.avg:.2f}% | top5={top5.avg:.2f}%"
            )

    return {
        "loss": objs.avg,
        "top1": top1.avg,
        "top5": top5.avg,
    }

In [33]:
if RUN_QUICK_CHECKS and RUN_SEARCH_EXAMPLE:
    # Prueba rápida del entrenamiento final desde cero

    # 1) Construir loaders finales
    final_train_queue, final_val_queue, final_test_queue = build_eval_dataloaders(
        EVAL_CFG,
        use_validation_split=True
    )

    # 2) Construir componentes finales
    final_components = build_final_components(
        eval_cfg=EVAL_CFG,
        genotype=best_genotype,
        num_classes=10
    )

    final_criterion = final_components["criterion"]
    final_model = final_components["model"]
    final_optimizer = final_components["optimizer"]
    final_scheduler = final_components["scheduler"]

    # 3) Ejecutar 1 época de entrenamiento final
    train_final_metrics = train_final_epoch(
        train_q=final_train_queue,
        model=final_model,
        criterion=final_criterion,
        optimizer=final_optimizer,
        eval_cfg=EVAL_CFG,
        epoch=0
    )

    # 4) Evaluación rápida sobre validation
    val_final_metrics = evaluate_final(
        data_q=final_val_queue,
        model=final_model,
        criterion=final_criterion,
        eval_cfg=EVAL_CFG,
        split_name="val-final"
    )

    print("\nResumen prueba rápida FINAL:")
    print(train_final_metrics)
    print(val_final_metrics)

## **14. Ejecutar evaluación final**

In [34]:
def run_evaluation(eval_cfg, genotype, algo_tag="first_order", num_classes=10):
    """
    Ejecuta el entrenamiento final desde cero de la arquitectura discreta
    y selecciona el mejor checkpoint por VALIDATION.
    El TEST se evalúa solo una vez al final.
    """
    set_reproducibility(eval_cfg["seed"], deterministic=False)

    run_dir = os.path.join(
        "results",
        "cifar10",
        "evaluation",
        algo_tag,
        f"seed_{eval_cfg['seed']}"
    )
    os.makedirs(run_dir, exist_ok=True)

    # Guardar configuración
    with open(os.path.join(run_dir, "config_eval.json"), "w") as f:
        json.dump(eval_cfg, f, indent=2)

    # Guardar genotipo usado
    with open(os.path.join(run_dir, "genotype_used.json"), "w") as f:
        json.dump(genotype_to_dict(genotype), f, indent=2)

    with open(os.path.join(run_dir, "genotype_used.txt"), "w") as f:
        f.write(str(genotype))

    # Dataloaders
    train_queue, val_queue, test_queue = build_eval_dataloaders(
        eval_cfg,
        use_validation_split=True
    )

    # Componentes finales
    final_components = build_final_components(
        eval_cfg=eval_cfg,
        genotype=genotype,
        num_classes=num_classes
    )

    criterion = final_components["criterion"]
    model = final_components["model"]
    optimizer = final_components["optimizer"]
    scheduler = final_components["scheduler"]

    history = []
    best_val_top1 = -1.0
    t0 = time.time()

    print("\n" + "=" * 70)
    print(f"FINAL EVALUATION — CIFAR-10 | {algo_tag} | seed={eval_cfg['seed']}")
    print("=" * 70)
    print(f"Epochs: {eval_cfg['epochs']}")
    print(f"Init channels: {eval_cfg['init_channels']}")
    print(f"Layers: {eval_cfg['layers']}")
    print("=" * 70)

    for epoch in range(eval_cfg["epochs"]):
        epoch_start = time.time()
        lr = scheduler.get_last_lr()[0]

        print(f"\nEpoch {epoch+1}/{eval_cfg['epochs']} | LR={lr:.6f}")
        print("-" * 70)

        train_metrics = train_final_epoch(
            train_q=train_queue,
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            eval_cfg=eval_cfg,
            epoch=epoch
        )

        val_metrics = evaluate_final(
            data_q=val_queue,
            model=model,
            criterion=criterion,
            eval_cfg=eval_cfg,
            split_name="val-final"
        )

        scheduler.step()
        epoch_time = time.time() - epoch_start

        epoch_record = {
            "epoch": epoch + 1,
            "lr": float(lr),
            "train_loss": float(train_metrics["loss"]),
            "train_top1": float(train_metrics["top1"]),
            "train_top5": float(train_metrics["top5"]),
            "val_loss": float(val_metrics["loss"]),
            "val_top1": float(val_metrics["top1"]),
            "val_top5": float(val_metrics["top5"]),
            "epoch_time_sec": float(epoch_time),
        }
        history.append(epoch_record)

        print(f"Train | loss={train_metrics['loss']:.4f} | top1={train_metrics['top1']:.2f}% | top5={train_metrics['top5']:.2f}%")
        print(f"Val   | loss={val_metrics['loss']:.4f} | top1={val_metrics['top1']:.2f}% | top5={val_metrics['top5']:.2f}%")
        print(f"Time  | {epoch_time:.1f} sec")

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]

            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_top1": best_val_top1,
                    "genotype": genotype_to_dict(genotype),
                },
                os.path.join(run_dir, "best_final_checkpoint.pt")
            )

            print(f"★ Nuevo mejor checkpoint final | val_top1={best_val_top1:.2f}%")

        with open(os.path.join(run_dir, "evaluation_history.json"), "w") as f:
            json.dump(history, f, indent=2)

    print("\nEvaluación final en TEST con el mejor modelo...")
    checkpoint = torch.load(os.path.join(run_dir, "best_final_checkpoint.pt"))
    model.load_state_dict(checkpoint["model_state_dict"])

    final_test_metrics = evaluate_final(
        data_q=test_queue,
        model=model,
        criterion=criterion,
        eval_cfg=eval_cfg,
        split_name="test-final"
    )

    total_time = time.time() - t0

    summary = {
        "algo_tag": algo_tag,
        "seed": int(eval_cfg["seed"]),
        "best_val_top1": float(best_val_top1),
        "final_test_top1": float(final_test_metrics["top1"]),
        "final_test_top5": float(final_test_metrics["top5"]),
        "final_test_loss": float(final_test_metrics["loss"]),
        "total_eval_time_sec": float(total_time),
        "run_dir": run_dir,
        "genotype": genotype_to_dict(genotype),
    }

    with open(os.path.join(run_dir, "evaluation_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n" + "=" * 70)
    print("FINAL EVALUATION TERMINADA")
    print("=" * 70)
    print(f"Best val top1: {best_val_top1:.2f}%")
    print(f"Final test top1: {final_test_metrics['top1']:.2f}%")
    print(f"Tiempo total: {total_time/60:.2f} min")
    print(f"Resultados en: {run_dir}")
    print("=" * 70)

    return summary

In [35]:
if RUN_EVAL_EXAMPLE and RUN_SEARCH_EXAMPLE:
    # Ejemplo de corrida de evaluación final
    eval_summary = run_evaluation(
        eval_cfg=EVAL_CFG,
        genotype=best_genotype,
        algo_tag="first_order",
        num_classes=10
    )

    print("\nResumen de evaluación final:")
    print(eval_summary)

## **15. Visualizar genotipo**

In [36]:
from graphviz import Digraph

def plot(genotype_edges, filename):
    g = Digraph(
        format='png',
        edge_attr=dict(fontsize='20', fontname="times"),
        node_attr=dict(
            style='filled',
            shape='rect',
            align='center',
            fontsize='20',
            height='0.5',
            width='0.5',
            penwidth='2',
            fontname="times"
        ),
        engine='dot'
    )
    g.body.extend(['rankdir=LR'])

    g.node("c_{k-2}", fillcolor='darkseagreen2')
    g.node("c_{k-1}", fillcolor='darkseagreen2')

    steps = len(genotype_edges) // 2
    for i in range(steps):
        g.node(str(i), fillcolor='lightblue')

    for i in range(steps):
        for k in range(2):
            op, j = genotype_edges[2 * i + k]
            if j == 0:
                u = "c_{k-2}"
            elif j == 1:
                u = "c_{k-1}"
            else:
                u = str(j - 2)
            v = str(i)
            g.edge(u, v, label=op, fillcolor="gray")

    g.node("c_k", fillcolor='palegoldenrod')
    for i in range(steps):
        g.edge(str(i), "c_k", fillcolor="gray")

    g.render(filename, view=False)

In [37]:
# Visualización del genotipo desde archivos

def load_genotype_from_json(genotype_json_path):
    with open(genotype_json_path, "r") as f:
        genotype_dict = json.load(f)
    return dict_to_genotype(genotype_dict)

In [38]:
def render_genotype_graphs(genotype_json_path, output_dir=None, prefix="best_genotype"):
    """
    Carga un genotipo desde JSON y genera:
    - grafo de la celda normal
    - grafo de la celda de reducción
    """
    genotype = load_genotype_from_json(genotype_json_path)

    if output_dir is None:
        output_dir = os.path.dirname(genotype_json_path)

    os.makedirs(output_dir, exist_ok=True)

    normal_path = os.path.join(output_dir, f"{prefix}_normal")
    reduce_path = os.path.join(output_dir, f"{prefix}_reduce")

    plot(genotype.normal, normal_path)
    plot(genotype.reduce, reduce_path)

    result = {
        "genotype_json_path": genotype_json_path,
        "normal_graph_png": normal_path + ".png",
        "reduce_graph_png": reduce_path + ".png",
    }

    print("=" * 70)
    print("GENOTYPE GRAPHS RENDERED")
    print("=" * 70)
    print(f"Genotype source: {genotype_json_path}")
    print(f"Normal cell graph:  {result['normal_graph_png']}")
    print(f"Reduce cell graph:  {result['reduce_graph_png']}")
    print("=" * 70)

    return result

## **16. Análisis final del modelo evaluado**

In [39]:
def load_final_model_from_checkpoint(eval_cfg, checkpoint_path, genotype_json_path, num_classes=10):
    genotype = load_genotype_from_json(genotype_json_path)

    model = build_final_model(
        eval_cfg=eval_cfg,
        genotype=genotype,
        num_classes=num_classes
    )

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, genotype, checkpoint

In [40]:
@torch.no_grad()
def collect_test_predictions(model, data_q):
    all_preds = []
    all_labels = []
    all_probs = []

    model.eval()

    for inp, tgt in data_q:
        inp = inp.to(DEVICE, non_blocking=True)

        logits = model(inp)
        if isinstance(logits, tuple):
            logits = logits[0]

        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.append(preds)
        all_labels.append(tgt.numpy())
        all_probs.append(probs.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    return all_preds, all_labels, all_probs

In [41]:
def analyze_final_model(eval_cfg, algo_tag="first_order", seed=None, num_classes=10):
    """
    Carga el mejor checkpoint final guardado por run_evaluation(...)
    y genera métricas y gráficas sobre el conjunto de test.
    """
    if seed is None:
        seed = eval_cfg["seed"]

    eval_dir = os.path.join(
        "results",
        "cifar10",
        "evaluation",
        algo_tag,
        f"seed_{seed}"
    )

    checkpoint_path = os.path.join(eval_dir, "best_final_checkpoint.pt")
    genotype_json_path = os.path.join(eval_dir, "genotype_used.json")
    summary_path = os.path.join(eval_dir, "evaluation_summary.json")

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"No existe checkpoint final: {checkpoint_path}")

    if not os.path.exists(genotype_json_path):
        raise FileNotFoundError(f"No existe genotype JSON: {genotype_json_path}")

    # Cargar modelo final
    model, genotype, checkpoint = load_final_model_from_checkpoint(
        eval_cfg=eval_cfg,
        checkpoint_path=checkpoint_path,
        genotype_json_path=genotype_json_path,
        num_classes=num_classes
    )

    # Construir test loader limpio
    _, _, test_queue = build_eval_dataloaders(eval_cfg, use_validation_split=True)

    # Recolectar predicciones
    all_preds, all_labels, all_probs = collect_test_predictions(model, test_queue)

    test_acc = float((all_preds == all_labels).mean() * 100.0)

    class_names = [
        'airplane', 'automobile', 'bird', 'cat', 'deer',
        'dog', 'frog', 'horse', 'ship', 'truck'
    ]

    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

    report_dict = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4,
        output_dict=True
    )

    report_text = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4
    )

    # Guardar matriz normalizada
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    im0 = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_title('Confusion Matrix (Normalized) — TEST')
    axes[0].set_xticks(range(10))
    axes[0].set_yticks(range(10))
    axes[0].set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    axes[0].set_yticklabels(class_names, fontsize=8)

    for i in range(10):
        for j in range(10):
            axes[0].text(
                j, i, f"{cm_norm[i, j]:.2f}",
                ha='center', va='center',
                color='white' if cm_norm[i, j] > 0.5 else 'black',
                fontsize=7
            )
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(cm, cmap='Greens')
    axes[1].set_title('Confusion Matrix (Counts) — TEST')
    axes[1].set_xticks(range(10))
    axes[1].set_yticks(range(10))
    axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    axes[1].set_yticklabels(class_names, fontsize=8)

    for i in range(10):
        for j in range(10):
            axes[1].text(
                j, i, f"{cm[i, j]}",
                ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black',
                fontsize=7
            )
    plt.colorbar(im1, ax=axes[1])

    plt.tight_layout()

    cm_path = os.path.join(eval_dir, "confusion_matrix_test.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Accuracy por clase
    class_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-10)

    fig = plt.figure(figsize=(10, 4))
    plt.bar(range(10), class_acc * 100)
    plt.xticks(range(10), class_names, rotation=45)
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy por clase — TEST")
    for i, acc in enumerate(class_acc * 100):
        plt.text(i, acc + 1, f"{acc:.1f}%", ha='center', fontsize=8)
    plt.grid(axis='y', alpha=0.3)

    acc_path = os.path.join(eval_dir, "class_accuracy_test.png")
    plt.savefig(acc_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Confianza por clase
    class_probs = [[] for _ in range(10)]
    for i in range(len(all_labels)):
        label = all_labels[i]
        class_probs[label].append(all_probs[i][label])

    confidence_summary = {}
    for i, class_name in enumerate(class_names):
        if len(class_probs[i]) > 0:
            mean_p = float(np.mean(class_probs[i]))
            std_p = float(np.std(class_probs[i]))
            conf = float(mean_p - std_p)
        else:
            mean_p = std_p = conf = None

        confidence_summary[class_name] = {
            "mean_prob": mean_p,
            "std_prob": std_p,
            "confidence_score": conf
        }

    analysis_summary = {
        "algo_tag": algo_tag,
        "seed": seed,
        "test_accuracy": test_acc,
        "confusion_matrix_png": cm_path,
        "class_accuracy_png": acc_path,
        "classification_report": report_dict,
        "confidence_summary": confidence_summary,
        "checkpoint_path": checkpoint_path,
        "genotype_json_path": genotype_json_path,
    }

    analysis_json_path = os.path.join(eval_dir, "analysis_summary.json")
    with open(analysis_json_path, "w") as f:
        json.dump(analysis_summary, f, indent=2)

    report_txt_path = os.path.join(eval_dir, "classification_report_test.txt")
    with open(report_txt_path, "w") as f:
        f.write(report_text)

    print("=" * 70)
    print("FINAL MODEL ANALYSIS")
    print("=" * 70)
    print(f"Evaluation dir: {eval_dir}")
    print(f"Test accuracy: {test_acc:.2f}%")
    print(f"Confusion matrix saved: {cm_path}")
    print(f"Class accuracy plot saved: {acc_path}")
    print(f"Classification report saved: {report_txt_path}")
    print(f"Analysis summary saved: {analysis_json_path}")
    print("=" * 70)

    print("\nClassification Report (TEST):\n")
    print(report_text)

    return analysis_summary

## **17. Corridas multisemilla**

In [42]:
def clone_cfg_with_seed(cfg, seed):
    cfg_new = copy.deepcopy(cfg)
    cfg_new["seed"] = seed
    return cfg_new

In [43]:
def run_multi_seed_experiments(
    base_search_cfg,
    base_eval_cfg,
    seeds,
    algo_tag="first_order",
    num_classes=10
):
    """
    Ejecuta el pipeline completo para múltiples semillas y
    guarda un resumen agregado en JSON.
    """
    all_results = []

    multi_dir = os.path.join(
        "results",
        "cifar10",
        "multi_seed",
        algo_tag
    )
    os.makedirs(multi_dir, exist_ok=True)

    print("\n" + "#" * 80)
    print(f"INICIO DE CORRIDAS MULTISEMILLA | {algo_tag}")
    print("#" * 80)
    print(f"Seeds: {seeds}")

    for seed in seeds:
        print("\n" + "=" * 80)
        print(f"SEED {seed}")
        print("=" * 80)

        search_cfg = clone_cfg_with_seed(base_search_cfg, seed)
        eval_cfg = clone_cfg_with_seed(base_eval_cfg, seed)

        full_summary = run_full_experiment(
            search_cfg=search_cfg,
            eval_cfg=eval_cfg,
            algo_tag=algo_tag,
            num_classes=num_classes
        )

        row = {
            "seed": seed,
            "algo_tag": algo_tag,
            "search_best_valid_top1": full_summary["search"]["best_valid_top1"],
            "search_time_sec": full_summary["search"]["total_search_time_sec"],
            "eval_best_val_top1": full_summary["evaluation"]["best_val_top1"],
            "eval_final_test_top1": full_summary["evaluation"]["final_test_top1"],
            "eval_final_test_top5": full_summary["evaluation"]["final_test_top5"],
            "eval_final_test_loss": full_summary["evaluation"]["final_test_loss"],
            "eval_time_sec": full_summary["evaluation"]["total_eval_time_sec"],
            "search_run_dir": full_summary["search"]["run_dir"],
            "eval_run_dir": full_summary["evaluation"]["run_dir"],
        }

        all_results.append(row)

        # Guardado incremental
        with open(os.path.join(multi_dir, "multi_seed_results.json"), "w") as f:
            json.dump(all_results, f, indent=2)

    # Resumen agregado
    test_top1_values = [r["eval_final_test_top1"] for r in all_results]
    val_top1_values = [r["eval_best_val_top1"] for r in all_results]
    search_valid_values = [r["search_best_valid_top1"] for r in all_results]

    aggregate_summary = {
        "algo_tag": algo_tag,
        "seeds": seeds,
        "n_runs": len(all_results),
        "search_best_valid_top1_mean": float(np.mean(search_valid_values)),
        "search_best_valid_top1_std": float(np.std(search_valid_values, ddof=1)) if len(search_valid_values) > 1 else 0.0,
        "eval_best_val_top1_mean": float(np.mean(val_top1_values)),
        "eval_best_val_top1_std": float(np.std(val_top1_values, ddof=1)) if len(val_top1_values) > 1 else 0.0,
        "eval_final_test_top1_mean": float(np.mean(test_top1_values)),
        "eval_final_test_top1_std": float(np.std(test_top1_values, ddof=1)) if len(test_top1_values) > 1 else 0.0,
        "runs": all_results,
    }

    with open(os.path.join(multi_dir, "multi_seed_summary.json"), "w") as f:
        json.dump(aggregate_summary, f, indent=2)

    print("\n" + "#" * 80)
    print("CORRIDAS MULTISEMILLA TERMINADAS")
    print("#" * 80)
    print(f"Mean final test top1: {aggregate_summary['eval_final_test_top1_mean']:.4f}")
    print(f"Std  final test top1: {aggregate_summary['eval_final_test_top1_std']:.4f}")
    print(f"Resultados en: {multi_dir}")
    print("#" * 80)

    return aggregate_summary

In [44]:
if RUN_MULTI_SEED:
    multi_seed_summary = run_multi_seed_experiments(
        base_search_cfg=SEARCH_CFG,
        base_eval_cfg=EVAL_CFG,
        seeds=EXPERIMENT_SEEDS,
        algo_tag="first_order",
        num_classes=10
    )

    print("\nResumen multisemilla:")
    print(json.dumps(multi_seed_summary, indent=2))

## **18. Ejecución integral del experimento**

In [45]:
def run_full_experiment(search_cfg, eval_cfg, algo_tag="first_order", num_classes=10):
    """
    Ejecuta el pipeline completo:
    1) búsqueda de arquitectura
    2) reconstrucción del mejor genotipo
    3) entrenamiento/evaluación final desde cero
    """
    print("\n" + "#" * 80)
    print("INICIO DEL EXPERIMENTO COMPLETO")
    print("#" * 80)

    # Fase 1: search
    search_summary = run_search(
        search_cfg=search_cfg,
        algo_tag=algo_tag,
        num_classes=num_classes
    )

    # Reconstrucción del mejor genotipo
    if search_summary["best_genotype"] is None:
        raise RuntimeError("Search no produjo un best_genotype válido.")

    best_genotype = dict_to_genotype(search_summary["best_genotype"])

    # Fase 2: evaluación final
    eval_summary = run_evaluation(
        eval_cfg=eval_cfg,
        genotype=best_genotype,
        algo_tag=algo_tag,
        num_classes=num_classes
    )

    full_summary = {
        "algo_tag": algo_tag,
        "search": search_summary,
        "evaluation": eval_summary,
    }

    out_dir = os.path.join(
        "results",
        "cifar10",
        "full_experiments",
        algo_tag,
        f"seed_{search_cfg['seed']}"
    )
    os.makedirs(out_dir, exist_ok=True)

    with open(os.path.join(out_dir, "full_experiment_summary.json"), "w") as f:
        json.dump(full_summary, f, indent=2)

    print("\n" + "#" * 80)
    print("EXPERIMENTO COMPLETO TERMINADO")
    print("#" * 80)
    print(json.dumps(full_summary, indent=2))

    return full_summary

In [46]:
def run_darts_comparison(
    eval_cfg,
    seeds,
    num_classes=10,
    run_analysis=True,
    render_genotype=True
):
    """
    Compara DARTS first-order vs second-order.
    """
    comparison_summary = {}

    print("\n" + "#" * 80)
    print("COMPARACIÓN DARTS: FIRST-ORDER VS SECOND-ORDER")
    print("#" * 80)
    print(f"Seeds: {seeds}")
    print("#" * 80)

    # First-order
    print("\n" + "=" * 80)
    print("FIRST-ORDER")
    print("=" * 80)

    first_order_results = run_multi_seed_experiments(
        base_search_cfg=SEARCH_CFG_FIRST_ORDER,
        base_eval_cfg=eval_cfg,
        seeds=seeds,
        algo_tag="first_order",
        num_classes=num_classes
    )

    comparison_summary["first_order"] = first_order_results

    # Second-order
    print("\n" + "=" * 80)
    print("SECOND-ORDER")
    print("=" * 80)

    second_order_results = run_multi_seed_experiments(
        base_search_cfg=SEARCH_CFG_SECOND_ORDER,
        base_eval_cfg=eval_cfg,
        seeds=seeds,
        algo_tag="second_order",
        num_classes=num_classes
    )

    comparison_summary["second_order"] = second_order_results

    # Resumen comparativo simple
    fo_mean = first_order_results["eval_final_test_top1_mean"]
    fo_std = first_order_results["eval_final_test_top1_std"]
    so_mean = second_order_results["eval_final_test_top1_mean"]
    so_std = second_order_results["eval_final_test_top1_std"]

    comparison_summary["comparison"] = {
        "first_order_mean_test_top1": fo_mean,
        "first_order_std_test_top1": fo_std,
        "second_order_mean_test_top1": so_mean,
        "second_order_std_test_top1": so_std,
        "difference_mean_test_top1": so_mean - fo_mean
    }

    out_dir = os.path.join("results", "cifar10", "comparisons")
    os.makedirs(out_dir, exist_ok=True)

    with open(os.path.join(out_dir, "darts_first_vs_second_order.json"), "w") as f:
        json.dump(comparison_summary, f, indent=2)

    print("\n" + "#" * 80)
    print("COMPARACIÓN TERMINADA")
    print("#" * 80)
    print(json.dumps(comparison_summary["comparison"], indent=2))

    return comparison_summary

In [47]:
if RUN_FULL_EXPERIMENT:
    # Ejecución integral oficial
    full_summary = run_full_experiment(
        search_cfg=SEARCH_CFG,
        eval_cfg=EVAL_CFG,
        algo_tag="first_order",
        num_classes=10
    )

In [48]:
'''
Ejecuta TODO.
Se pueden agregar más semillas manualmente.
'''

comparison_summary = run_darts_comparison(
    eval_cfg=EVAL_CFG,
    seeds=[11],
    num_classes=10
)


################################################################################
COMPARACIÓN DARTS: FIRST-ORDER VS SECOND-ORDER
################################################################################
Seeds: [11]
################################################################################

FIRST-ORDER

################################################################################
INICIO DE CORRIDAS MULTISEMILLA | first_order
################################################################################
Seeds: [11]

SEED 11

################################################################################
INICIO DEL EXPERIMENTO COMPLETO
################################################################################
[Reproducibility] Seed=11 | deterministic=False


/home/jabonsote/miniconda3/envs/vjepa2-312/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


SEARCH DATALOADERS
Train-search samples: 25000 | batches: 391
Valid-search samples: 25000 | batches: 391
SEARCH MODEL COMPONENTS
Model params: 1.931 MB
Init channels: 16
Layers: 8
Steps: 4
Multiplier: 4
Stem multiplier: 3
Unrolled: False

DARTS SEARCH — CIFAR-10 | first_order | seed=11
Epochs: 50
Init channels: 16
Layers: 8
Unrolled: False

Epoch 1/50 | LR=0.025000
----------------------------------------------------------------------
Genotype (epoch start): Genotype(normal=[('dil_conv_5x5', 1), ('dil_conv_5x5', 0), ('sep_conv_5x5', 2), ('dil_conv_5x5', 0), ('sep_conv_5x5', 2), ('dil_conv_3x3', 3), ('avg_pool_3x3', 3), ('sep_conv_5x5', 1)], normal_concat=[2, 3, 4, 5], reduce=[('avg_pool_3x3', 1), ('dil_conv_3x3', 0), ('skip_connect', 2), ('dil_conv_3x3', 0), ('avg_pool_3x3', 1), ('dil_conv_5x5', 0), ('sep_conv_3x3', 1), ('dil_conv_5x5', 3)], reduce_concat=[2, 3, 4, 5])
  [train-search] Step 000/391 | loss=2.3352 | top1=10.94% | top5=46.88%
  [train-search] Step 050/391 | loss=2.0224 | 

KeyboardInterrupt: 

In [ ]:
'''
Si se quiere ejecutar, cambiar `RUN_GENOTYPE_VIS` a `True`.
Se ejecuta por semilla, se puede hacer en las semillas de mayor interés.
'''

RUN_GENOTYPE_VIS = False
if RUN_GENOTYPE_VIS:
    genotype_json_path = os.path.join(
        "results",
        "cifar10",
        "search",
        "first_order",
        f"seed_{SEARCH_CFG['seed']}",
        "best_genotype.json"
    )

    genotype_vis_summary = render_genotype_graphs(
        genotype_json_path=genotype_json_path,
        output_dir=os.path.dirname(genotype_json_path),
        prefix="best_genotype"
    )

    print("\nResumen visualización genotipo:")
    print(json.dumps(genotype_vis_summary, indent=2))

GENOTYPE GRAPHS RENDERED
Genotype source: results/cifar10/search/first_order/seed_11/best_genotype.json
Normal cell graph:  results/cifar10/search/first_order/seed_11/best_genotype_normal.png
Reduce cell graph:  results/cifar10/search/first_order/seed_11/best_genotype_reduce.png

Resumen visualización genotipo:
{
  "genotype_json_path": "results/cifar10/search/first_order/seed_11/best_genotype.json",
  "normal_graph_png": "results/cifar10/search/first_order/seed_11/best_genotype_normal.png",
  "reduce_graph_png": "results/cifar10/search/first_order/seed_11/best_genotype_reduce.png"
}


In [ ]:
'''
Si se quiere ejecutar, cambiar `RUN_FINAL-ANALYSIS` a `True`.
Se ejecuta por semilla, se puede hacer en las semillas de mayor interés.
'''

RUN_FINAL_ANALYSIS = False
if RUN_FINAL_ANALYSIS:
    # Prueba rápida
    analysis_summary = analyze_final_model(
        eval_cfg=EVAL_CFG,
        algo_tag="first_order",
        seed=EVAL_CFG["seed"],
        num_classes=10
    )

    print("\nResumen análisis final:")
    print(json.dumps(analysis_summary, indent=2))

## Especificaciones de hardware y software

In [ ]:
import platform
import psutil

def report_system_info():
    print(f"Sistema Operativo: {platform.system()} {platform.release()}")
    print(f"Procesador: {platform.processor()}")
    print(f"Velocidad CPU: {psutil.cpu_freq().current:.2f} MHz")
    print(f"Memoria RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")
    print(f"Lenguaje: Python {platform.python_version()}")
    # En Google Colab:
    !gcc --version | head -n 1

In [ ]:
if RUN_SYSTEM_INFO:
  # Ver las especificaciones de hardware y software
  report_system_info()